## Setup

### Constants

In [ ]:
FILTER_WARNINGS = False

### Constants (require import)

In [ ]:
from pathlib import Path
CLUSTER01_DIR = Path("./cluster01")
SG_USER_CLUSTER01_CSV = CLUSTER01_DIR / "sg_user.csv"

In [ ]:
import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.DEBUG)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

def set_mckinsey_style():
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
    global MCK_BLUE, MCK_LIGHT_BLUE, MCK_GREY, MCK_ACCENT
    MCK_BLUE = '#002060'       # 深蓝 (Core/Singapore)
    MCK_LIGHT_BLUE = '#00A9E0' # 亮蓝 (Growth/Malaysia)
    MCK_GREY = '#7F7F7F'       # 灰色 (辅助/Passive)
    MCK_ACCENT = '#E4002B'     # 红色 (重点/Highlight)

    sns.set_context("talk")
    sns.set_style("white")
    plt.rcParams['axes.edgecolor'] = '#d9d9d9'
    plt.rcParams['axes.linewidth'] = 1
    plt.rcParams['xtick.color'] = '#555555'
    plt.rcParams['ytick.color'] = '#555555'
    plt.rcParams['text.color'] = '#333333'
    plt.rcParams['axes.labelcolor'] = '#333333'
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['axes.labelsize'] = 12
set_mckinsey_style()

### Imports and Setting

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

if FILTER_WARNINGS:
    import warnings
    warnings.filterwarnings("ignore")


plt.set_loglevel('warning')


logging.info("Setup completed successfully")

## Load dataset

In [ ]:
logging.debug("Loading dataset")

try:
    customer_features = pd.read_csv(SG_USER_CLUSTER01_CSV)


    logging.info(f"Customer Features: {len(customer_features):,} rows")
except FileNotFoundError as e:
    logging.error(f"Error csv not found: {e}")
    raise
except Exception as e:
    logging.critical(f"Load csv error: {e}")
    raise

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load the Singapore data
df_sg = pd.read_csv('./cluster01/sg_user.csv')

def set_mckinsey_style():
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
    global MCK_BLUE, MCK_LIGHT_BLUE, MCK_GREY, MCK_ACCENT
    MCK_BLUE = '#002060'       # 深蓝 (Core/Singapore)
    MCK_LIGHT_BLUE = '#00A9E0' # 亮蓝 (Growth/Malaysia)
    MCK_GREY = '#7F7F7F'       # 灰色 (辅助/Passive)
    MCK_ACCENT = '#E4002B'     # 红色 (重点/Highlight)

    sns.set_context("talk")
    sns.set_style("white")
    plt.rcParams['axes.edgecolor'] = '#d9d9d9'
    plt.rcParams['axes.linewidth'] = 1
    plt.rcParams['xtick.color'] = '#555555'
    plt.rcParams['ytick.color'] = '#555555'
    plt.rcParams['text.color'] = '#333333'
    plt.rcParams['axes.labelcolor'] = '#333333'
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['axes.labelsize'] = 12

set_mckinsey_style()

# ============================================================
# PLOT 1: Customer Distribution by Value Cluster (SG)
# ============================================================

# 1. Count customers per cluster, sorted by count descending
segment_counts = df_sg['value_cluster'].value_counts().sort_values(ascending=False)

# 2. Create figure
fig = plt.figure(figsize=(14, 8))
ax = plt.gca()

# 3. Horizontal bar plot
plot = sns.barplot(y=segment_counts.index.astype(str), x=segment_counts.values, 
                   color=MCK_BLUE, ax=ax)  # Using MCK_BLUE for Singapore

# 4. Add title and labels
plt.suptitle('Customer Distribution by Value Cluster (SG)', 
             fontsize=20, fontweight='bold', color=MCK_BLUE, y=0.98)
plt.xlabel('Number of Customers', fontsize=13, color=MCK_GREY, fontweight='normal')
plt.ylabel('Value Cluster', fontsize=13, color=MCK_GREY, fontweight='normal')

# 5. Add value labels at end of bars
for i, v in enumerate(segment_counts.values):
    plt.text(v + (max(segment_counts.values)*0.02), i, str(v), 
             va='center', fontsize=12, color=MCK_BLUE, fontweight='bold')

# 6. Clean styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='x', linestyle=':', alpha=0.3, color=MCK_GREY)

# 7. Show plot
plt.tight_layout()
plt.show()

# ============================================================
# PLOT 2: Taste Profile Composition by Value Cluster (SG)
# ============================================================

set_mckinsey_style()

# Filter out rows where taste preferences are unknown (all equal to 0.5)
# This assumes unknown preferences are marked as 0.5 for all three tastes
df_sg_cleaned = df_sg[~((df_sg['sweets'] == 0.5) & 
                        (df_sg['fruits'] == 0.5) & 
                        (df_sg['flower'] == 0.5))].copy()

# Calculate the sum of preferences for each cluster
taste_by_cluster = df_sg_cleaned.groupby('value_cluster')[['sweets', 'fruits', 'flower']].sum()

# Calculate proportional percentages (sums to 100% per cluster)
taste_percentages = taste_by_cluster.div(taste_by_cluster.sum(axis=1), axis=0) * 100

# Sort by cluster number
taste_percentages = taste_percentages.sort_index()

# Get cluster sizes for annotation
cluster_totals = df_sg_cleaned.groupby('value_cluster').size()

fig, ax = plt.subplots(figsize=(14, 8))

# Create horizontal stacked bar chart
taste_percentages.plot(kind='barh', stacked=True, ax=ax,
                       color=[MCK_BLUE, MCK_LIGHT_BLUE, MCK_GREY],
                       width=0.75, edgecolor='white', linewidth=1.5)

# Add percentage labels on each segment
for i, cluster in enumerate(taste_percentages.index):
    cumulative = 0
    for taste in ['sweets', 'fruits', 'flower']:
        value = taste_percentages.loc[cluster, taste]
        if value > 8:  # Only show label if segment is large enough
            ax.text(cumulative + value/2, i, f'{value:.0f}%',
                   ha='center', va='center', fontsize=11, 
                   color='white', fontweight='bold')
        cumulative += value

# Styling
plt.suptitle('Known Taste Profile Composition by Value Cluster (SG)', 
             fontsize=20, fontweight='bold', color=MCK_BLUE, y=0.98)
ax.set_xlabel('Proportion of Taste Preferences (%)', fontsize=17, color=MCK_GREY)
ax.set_ylabel('Value Cluster', fontsize=17, color=MCK_GREY)
ax.set_xlim(0, 100)

# Add sample size annotation
for i, cluster in enumerate(taste_percentages.index):
    n = cluster_totals[cluster]
    ax.text(102, i, f'n={n}', va='center', fontsize=10, color=MCK_GREY)

# Legend - positioned next to x-axis label at bottom center
ax.legend(['Sweets', 'Fruits', 'Flower'], 
          loc='upper center', bbox_to_anchor=(0.5, -0.12), 
          ncol=3, frameon=True, 
          fancybox=False, edgecolor='#d9d9d9', fontsize=12)

# Remove spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='x', linestyle=':', alpha=0.3, color=MCK_GREY)

plt.tight_layout()
plt.show()